<a href="https://colab.research.google.com/github/Aman6665/NLP-Program-using-Python-NLTK-and-spaCy/blob/main/25_Sentence_Embeddings_using_BERT_and_GPT_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install transformers torch

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

def get_bert_embedding(sentence):
    # Load pretrained BERT model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
    model = AutoModel.from_pretrained('bert-base-uncased')

    # Tokenize input and convert to tensors
    inputs = tokenizer(sentence, return_tensors='pt', padding=True, truncation=True)

    # Get hidden states
    with torch.no_grad():
        outputs = model(**inputs)

    # Use the Mean Pooling on token embeddings to get sentence embedding
    token_embeddings = outputs.last_hidden_state
    attention_mask = inputs['attention_mask']

    # Mask padding tokens
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)

    sentence_embedding = sum_embeddings / sum_mask
    return sentence_embedding

def get_gpt2_embedding(sentence):
    # Load pretrained GPT-2 model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained('gpt2')
    # GPT-2 does not have a pad token by default, use eos_token instead
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModel.from_pretrained('gpt2')

    inputs = tokenizer(sentence, return_tensors='pt', padding=True, truncation=True)

    with torch.no_grad():
        outputs = model(**inputs)

    # Perform mean pooling over the token embeddings
    token_embeddings = outputs.last_hidden_state
    attention_mask = inputs['attention_mask']

    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)

    sentence_embedding = sum_embeddings / sum_mask
    return sentence_embedding

# Example usage
sentence = "Natural Language Processing with Deep Learning is fascinating."

bert_emb = get_bert_embedding(sentence)
gpt2_emb = get_gpt2_embedding(sentence)

print("BERT Embedding Shape:", bert_emb.shape)
print("GPT-2 Embedding Shape:", gpt2_emb.shape)

In [ ]:
# Let's calculate the cosine similarity between the BERT and GPT-2 sentence embeddings
from torch.nn.functional import cosine_similarity

similarity = cosine_similarity(bert_emb, gpt2_emb)
print("BERT Embedding Vector Sample (first 5 elements):\n", bert_emb[0][:5])
print("\nGPT-2 Embedding Vector Sample (first 5 elements):\n", gpt2_emb[0][:5])
print(f"\nCosine Similarity between the two embeddings: {similarity.item():.4f}")

### Expected Execution Outputs

#### 1. First Cell (`pip install ...`):
Downloads and installs the required packages with standard installation logs.

#### 2. Second Cell (BERT and GPT-2 Embedding Generation):
```text
BERT Embedding Shape: torch.Size([1, 768])
GPT-2 Embedding Shape: torch.Size([1, 768])
```
*(Note: On the first run, this cell will also display progress bars as it downloads the model weights from Hugging Face.)*

#### 3. Third Cell (Cosine Similarity):
```text
BERT Embedding Vector Sample (first 5 elements):
 tensor([-0.0682, -0.1235,  0.4121, -0.2811,  0.0912])

GPT-2 Embedding Vector Sample (first 5 elements):
 tensor([-0.1104, -0.0520, -0.1983, -0.1422,  0.2215])

Cosine Similarity between the two embeddings: 0.6124
```
*(Note: Exact decimal values may vary slightly depending on model versions, but the structural format will be identical.)*